In [2]:
import fitz
import pytesseract
from PIL import Image
import numpy as np
import cv2
import os
import unicodedata
import re

%cd /media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

BASE_DIR = "Datasets/Unstructured_Data/8asl_amwal"

BIDI_RE = re.compile('[\u200E\u200F\u202A-\u202E\u2066-\u2069]')

def preprocess(img):
    img = np.array(img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.medianBlur(gray, 3)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(th)

def clean_text(text):
    text = BIDI_RE.sub('', text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'\u00A0', ' ', text)
    text = re.sub(r'[ ]{2,}', ' ', text)
    return text

# 🔥 Loop over all PDFs
for filename in os.listdir(BASE_DIR):
    
    if filename.lower().endswith(".pdf"):
        
        pdf_path = os.path.join(BASE_DIR, filename)
        output_txt = os.path.join(
            BASE_DIR,
            filename.replace(".pdf", "_ultra_clean.txt")
        )
        
        print(f"\n==============================")
        print(f"Processing file: {filename}")
        print(f"==============================")
        
        doc = fitz.open(pdf_path)
        all_text = ""
        
        for i, page in enumerate(doc):
            print(f"  Page {i+1}/{len(doc)}")
            
            mat = fitz.Matrix(4, 4)  # High DPI
            pix = page.get_pixmap(matrix=mat)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            
            img = preprocess(img)
            
            text = pytesseract.image_to_string(
                img,
                lang="ara",
                config="--psm 6"
            )
            
            text = clean_text(text)
            all_text += f"\n\n===== PAGE {i+1} =====\n\n"
            all_text += text
        
        with open(output_txt, "w", encoding="utf-8") as f:
            f.write(all_text)
        
        print(f"✔ Saved: {output_txt}")

print("\n🔥 All PDFs processed successfully.")

/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject

Processing file: قانون-مكافحة-غسل-الامول.pdf
  Page 1/8
  Page 2/8
  Page 3/8
  Page 4/8
  Page 5/8
  Page 6/8
  Page 7/8
  Page 8/8
✔ Saved: Datasets/Unstructured_Data/8asl_amwal/قانون-مكافحة-غسل-الامول_ultra_clean.txt

🔥 All PDFs processed successfully.
